# МТС · Qwen3.5-0.8B · настоящий LoRA-пилот

Запускать ячейки по очереди. Нужны два файла: **mts_train_ready_v2.zip** и **mts_ml_pilot_code.zip**. Это новый эксперимент; прежний smoke notebook не изменяется.

Пилот: 32 train / 8 dev, 64 шага, последние 50 шагов используются для оценки времени. Парное сравнение до/после — первые 2 train и 2 dev; после обучения дополнительно проверяются все 32/8. Это диагностика процесса, не репрезентативная оценка качества. Проверяются completion-only loss, память и повторная загрузка адаптера. До выполнения ячеек результатов обучения нет.


## 1. GPU и загрузка файлов
Выберите среду выполнения с GPU, затем загрузите оба ZIP.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile
import torch
from google.colab import files
assert torch.cuda.is_available(), "GPU отсутствует: выберите GPU в настройках Colab"
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)
uploaded = files.upload()
for filename in ("mts_train_ready_v2.zip", "mts_ml_pilot_code.zip"):
    assert filename in uploaded, f"Нужен файл {filename}"
    with zipfile.ZipFile(filename) as z:
        assert z.testzip() is None
        for item in z.infolist():
            target = (Path("/content") / item.filename).resolve()
            assert target.is_relative_to(Path("/content"))
        z.extractall("/content")
del uploaded
DATA = Path("/content/mts_train_ready_v2")
CODE = Path("/content/mts_ml_code")
subprocess.run([sys.executable, str(DATA/"verify.py")], check=True)
print("Данные и исходники распакованы и проверены")


## 2. Зависимости
Используется установленный CUDA PyTorch. Модель загружается в отдельном Python-процессе, чтобы избежать старых импортов ядра.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE/"ml/qwen/requirements.txt")], check=True)
subprocess.run([sys.executable, "-c", "import torch,peft,transformers;from transformers.models.qwen3_5 import Qwen3_5ForConditionalGeneration;print('Torch',torch.__version__,'Transformers',transformers.__version__,'PEFT',peft.__version__)"], check=True)


## 3. Пилот: реальное обучение
Сначала сохраняются ответы до обучения. Затем — loss/время/память, best/last, тест reload и ответы после. Длительность заранее неизвестна. Ошибка и пустой успешный ответ учитываются отдельно.

In [ ]:
from datetime import datetime, timezone
def stream_run(command, cwd):
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    except BaseException:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
        raise
RUN = Path("/content/mts_ml_runs") / datetime.now(timezone.utc).strftime("pilot_%Y%m%dT%H%M%SZ")
command = [sys.executable, "-u", "-m", "ml.qwen.train", "--data-root", str(DATA), "--out", str(RUN), "--phase", "pilot", "--steps", "64", "--precision", "float32", "--learning-rate", "5e-5"]
print("Результаты:", RUN)
stream_run(command, CODE)


## 4. Посмотреть и скачать результат
Скачайте архив до отключения среды Colab. В нём адаптер, tokenizer, logs, реальные predictions и метрики; базовые веса повторно скачиваются из закреплённого checkpoint.

In [ ]:
import shutil
report = json.loads((RUN/"pilot.json").read_text())
print(json.dumps(report, ensure_ascii=False, indent=2))
result_zip = shutil.make_archive(str(RUN), "zip", RUN)
files.download(result_zip)


## 5. Один основной SFT — только после успешного пилота
По умолчанию выключен, чтобы «Выполнить всё» не запустило длинную работу случайно. После проверки пилота установите RUN_MAIN=True. Проверяются реальные gates и ETA; обучение идёт на полном train, один epoch, затем полный официальный val до и после. Внутренний лимит времени обучения — 4 часа, это настройка эксперимента, не официальное правило МТС.

In [ ]:
RUN_MAIN = False
if RUN_MAIN:
    assert report.get("main_sft_ready"), "Сначала исправить причины непрохождения пилота"
    MAIN_RUN = RUN.parent / RUN.name.replace("pilot_", "main_", 1)
    stream_run([sys.executable, "-u", "-m", "ml.qwen.train", "--data-root", str(DATA), "--out", str(MAIN_RUN), "--phase", "main", "--pilot-report", str(RUN/"pilot.json"), "--budget-hours", "4", "--full-eval"], CODE)
    files.download(shutil.make_archive(str(MAIN_RUN), "zip", MAIN_RUN))
else:
    print("Основной SFT не запускался. Для запуска после успешного пилота установите RUN_MAIN=True.")
